In [ ]:
import ROOT
from ROOT import TChain, TLorentzVector 
import numpy as np
import warnings

In [ ]:
dir = "Data/"

mc_file_path_1 = dir + "reco_ntuple2_LMNR_1.root" # MC
mc_file_path_2 = dir + "reco_ntuple_LMNR_1.root"  # MC

data_file_path_1 = dir + "ntuple2_flat_LMNR_PostRefitMomenta_test_2022F_skimSoftMu_1.root" #Data
data_file_path_2 = dir + "ntuple_flat_LMNR_PostRefitMomenta_test_2022F_skimSoftMu_1.root"  #Data

# Criar TChains para Monte Carlo e Dados
print("Criando TChain para Monte Carlo...")
chain1 = ROOT.TChain("ntuple")
chain1.Add(mc_file_path_1)
chain1.Add(mc_file_path_2)

print("Criando TChain para Dados Experimentais...")
chain2 = ROOT.TChain("ntuple")
chain2.Add(data_file_path_1)
chain2.Add(data_file_path_2)

print(f"Total de entradas no TChain de Monte Carlo: {chain1.GetEntries()}")
print(f"Total de entradas no TChain de Dados: {chain2.GetEntries()}")

output_file1 = ROOT.TFile("MC.root", "RECREATE")
output_tree1 = ROOT.TTree("signal_tree", "Tree with selected variables from Monte Carlo")

output_file2 = ROOT.TFile("ED.root", "RECREATE")
output_tree2 = ROOT.TTree("background_tree", "Tree with selected variables from experimental data")

In [ ]:
variables = {
    "mumuMass": np.zeros(1, dtype=float),
    "bCosAlphaBS": np.zeros(1, dtype=float),
    "bVtxCL": np.zeros(1, dtype=float),
    "bLBSs": np.zeros(1, dtype=float),
    "bDCABSs": np.zeros(1, dtype=float),
    "kstTrkpDCABSs": np.zeros(1, dtype=float),
    "kstTrkmDCABSs": np.zeros(1, dtype=float),
    "bTMass": np.zeros(1, dtype=float),
    "kstTMass": np.zeros(1, dtype=float),
    "leadingPt": np.zeros(1, dtype=float),
    "trailingPt": np.zeros(1, dtype=float),
}

for var in variables:
    output_tree1.Branch(var, variables[var], f"{var}/D")
    output_tree2.Branch(var, variables[var], f"{var}/D")

In [ ]:
def TLvector(m, pt, eta, phi):
    lv = TLorentzVector()
    lv.SetPtEtaPhiM(pt, eta, phi, m)
    return lv

In [ ]:
def cyl_coord(particle, tree, i):
    C = []
    pt = getattr(tree, particle + "Pt")
    eta = getattr(tree, particle + "Eta")
    phi = getattr(tree, particle + "Phi")
    C.append(pt)
    C.append(eta)
    C.append(phi)
    return C

In [ ]:
def flavour_tag(Pm, Pp):
    PION_MASS = 0.13957018 # Massa do píon carregado
    KAON_MASS = 0.493677  # Massa do Kaon carregado
    KSTAR_MASS = 0.89594  # Massa do K*(892)

    lvm1 = TLvector(PION_MASS, Pm[0], Pm[1], Pm[2])  # π-
    lvp1 = TLvector(KAON_MASS, Pp[0], Pp[1], Pp[2])  # K+
    lvm2 = TLvector(KAON_MASS, Pm[0], Pm[1], Pm[2])  # K-
    lvp2 = TLvector(PION_MASS, Pp[0], Pp[1], Pp[2])  # π+

    m1 = (lvm1 + lvp1).M() # Massa invariante para combinação (π-, K+)
    m2 = (lvm2 + lvp2).M() # Massa invariante para combinação (K-, π+)

    if abs(m1 - KSTAR_MASS) < abs(m2 - KSTAR_MASS):
        fl_tag = 1  # K*
    else:
        fl_tag = 2  # K* Bar (anti-K*)
    return fl_tag

In [ ]:
def fill_tree(tree_chain, output_tree, is_data):
    print(f"\nPreenchendo {output_tree.GetName()} (is_data={is_data})...")
    for i in range(tree_chain.GetEntries()):
        tree_chain.GetEntry(i) 
        
        mumuMass = getattr(tree_chain, "mumuMass")
        bCosAlphaBS = getattr(tree_chain, "bCosAlphaBS")
        bVtxCL = getattr(tree_chain, "bVtxCL")
        bLBS = getattr(tree_chain, "bLBS")
        bLBSE = getattr(tree_chain, "bLBSE")
        bDCABS = getattr(tree_chain, "bDCABS")
        bDCABSE = getattr(tree_chain, "bDCABSE")
        kstTrkpDCABS = getattr(tree_chain, "kstTrkpDCABS")
        kstTrkpDCABSE = getattr(tree_chain, "kstTrkpDCABSE")
        kstTrkmDCABS = getattr(tree_chain, "kstTrkmDCABS")
        kstTrkmDCABSE = getattr(tree_chain, "kstTrkmDCABSE")
        bMass = getattr(tree_chain, "bMass")
        kstMass = getattr(tree_chain, "kstMass")
        bBarMass = getattr(tree_chain, "bBarMass")
        kstBarMass = getattr(tree_chain, "kstBarMass")
        mumPt = getattr(tree_chain, "mumPt")
        mupPt = getattr(tree_chain, "mupPt")

        Cm = cyl_coord("kstTrkm", tree_chain, i)
        Cp = cyl_coord("kstTrkp", tree_chain, i)

        fl_tag = flavour_tag(Cm, Cp)
        tmatch = True 
        
        SIGNAL_MASS_LOWER             = 5.133542769
        SIGNAL_MASS_UPPER             = 5.416657231
        BACKGROUND_MASS_LOWER_WINDOW1 = 5.0
        BACKGROUND_MASS_UPPER_WINDOW1 = 5.133542769
        BACKGROUND_MASS_LOWER_WINDOW2 = 5.416657231
        BACKGROUND_MASS_UPPER_WINDOW2 = 5.6

        # Classificar amostras como background ou sinal
        if fl_tag == 1: # K*
            background = is_data and (
                (bMass > BACKGROUND_MASS_LOWER_WINDOW1 and bMass < BACKGROUND_MASS_UPPER_WINDOW1) or
                (bMass > BACKGROUND_MASS_LOWER_WINDOW2 and bMass < BACKGROUND_MASS_UPPER_WINDOW2)
            )
            signal = not is_data and (bMass >= SIGNAL_MASS_LOWER and bMass <= SIGNAL_MASS_UPPER)
        elif fl_tag == 2: # K* Bar
            background = is_data and (
                (bBarMass > BACKGROUND_MASS_LOWER_WINDOW1 and bBarMass < BACKGROUND_MASS_UPPER_WINDOW1) or
                (bBarMass > BACKGROUND_MASS_LOWER_WINDOW2 and bBarMass < BACKGROUND_MASS_UPPER_WINDOW2)
            )
            signal = not is_data and (bBarMass >= SIGNAL_MASS_LOWER and bBarMass <= SIGNAL_MASS_UPPER)
        else:
            continue

        if not is_data:
            tMum = getattr(tree_chain, "truthMatchMum")
            tMup = getattr(tree_chain, "truthMatchMup")
            tTrkm = getattr(tree_chain, "truthMatchTrkm")
            tTrkp = getattr(tree_chain, "truthMatchTrkp")
            tmatch = (tMum and tMup and tTrkm and tTrkp)

        if tmatch and (signal or background):
            if fl_tag == 1:
                variables["bTMass"][0] = bMass
                variables["kstTMass"][0] = kstMass
            elif fl_tag == 2:
                variables["bTMass"][0] = bBarMass
                variables["kstTMass"][0] = kstBarMass

            variables["mumuMass"][0] = mumuMass
            variables["bCosAlphaBS"][0] = bCosAlphaBS
            variables["bVtxCL"][0] = bVtxCL

            if bLBSE != 0:
                variables["bLBSs"][0] = bLBS / bLBSE
            else:
                variables["bLBSs"][0] = 0.0 

            if bDCABSE != 0:
                variables["bDCABSs"][0] = bDCABS / bDCABSE
            else:
                variables["bDCABSs"][0] = 0.0

            if kstTrkpDCABSE != 0:
                variables["kstTrkpDCABSs"][0] = kstTrkpDCABS / kstTrkpDCABSE
            else:
                variables["kstTrkpDCABSs"][0] = 0.0

            if kstTrkmDCABSE != 0:
                variables["kstTrkmDCABSs"][0] = kstTrkmDCABS / kstTrkmDCABSE
            else:
                variables["kstTrkmDCABSs"][0] = 0.0

            if mumPt > mupPt:
                variables["leadingPt"][0] = mumPt
                variables["trailingPt"][0] = mupPt
            else:
                variables["leadingPt"][0] = mupPt
                variables["trailingPt"][0] = mumPt

            output_tree.Fill()
    print(f"Preenchimento de {output_tree.GetName()} concluído.")

In [ ]:
fill_tree(chain1, output_tree1, False) 
fill_tree(chain2, output_tree2, True)  

print("\nEscrevendo e fechando arquivos de saída...")
output_file1.Write()
output_file1.Close()

output_file2.Write()
output_file2.Close()